# 4주차. 실제 MCP 서버 Tool Call과 Gradio UI

## 0. 목표

로컬 FastMCP 서버를 만들고, MCP client adapter로 서버의 tool을 읽어와 실제 OpenAI Agent와 Gradio UI에서 호출한다.


## 1. 준비


In [ ]:
import json
import os
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

MCP는 외부 도구를 서버로 노출하고 client가 표준 protocol로 tool 목록과 실행 결과를 주고받게 해준다. 이번 주에는 노트북에서 작은 FastMCP 서버를 만들고, stdio transport로 연결해 LangChain agent가 실제 MCP tool을 호출하게 한다.


## 3. 기본 개념 실습

가장 작은 흐름은 로컬 MCP 서버를 만들고, MCP client가 stdio transport로 서버의 tool 목록을 읽어오는 것이다. 여기서는 실제 `FastMCP` 서버에 `calendar_check_availability` tool을 등록한다.


In [ ]:
import asyncio
import pathlib
import sys
import tempfile
import textwrap
import threading

from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_server_path = pathlib.Path(tempfile.gettempdir()) / "kanamate_calendar_mcp_server.py"


def run_async(coro):
    'Run an async MCP call from both notebooks and plain Python.'
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    box: dict[str, Any] = {}

    def runner() -> None:
        try:
            box["value"] = asyncio.run(coro)
        except Exception as exc:
            box["error"] = exc

    thread = threading.Thread(target=runner)
    thread.start()
    thread.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


def write_calendar_mcp_server(include_create_event: bool = False) -> pathlib.Path:
    'Write a small real MCP server that exposes calendar tools over stdio.'
    create_event_tool = '''

@mcp.tool()
def calendar_create_event(title: str, date: str, start_time: str, members: list[str]) -> dict:
    \'그룹 일정을 생성한다.\'
    return {
        \'server\': \'kanamate-calendar\',
        \'tool\': \'calendar.create_event\',
        \'arguments\': {\'title\': title, \'date\': date, \'start_time\': start_time, \'members\': members},
        \'event_id\': f"event-{date}-{start_time}".replace(\':\', \'\'),
        \'status\': \'created\',
    }
''' if include_create_event else ""

    server_code = r'''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP('kanamate-calendar')

@mcp.tool()
def calendar_check_availability(members: list[str], date: str) -> dict:
    '그룹 멤버의 가능한 시간을 조회한다.'
    return {
        'server': 'kanamate-calendar',
        'tool': 'calendar.check_availability',
        'arguments': {'members': members, 'date': date},
        'available_slots': [f'{date} 10:00', f'{date} 15:00'],
    }
# CREATE_EVENT_TOOL

if __name__ == '__main__':
    mcp.run(transport='stdio')
'''.replace("# CREATE_EVENT_TOOL", create_event_tool)
    mcp_server_path.write_text(textwrap.dedent(server_code).lstrip(), encoding="utf-8")
    return mcp_server_path


def make_calendar_mcp_client(server_path: pathlib.Path) -> MultiServerMCPClient:
    return MultiServerMCPClient(
        {
            "calendar": {
                "command": sys.executable,
                "args": [str(server_path)],
                "transport": "stdio",
            }
        }
    )


def load_calendar_mcp_tools() -> list[Any]:
    client = make_calendar_mcp_client(mcp_server_path)
    return run_async(client.get_tools(server_name="calendar"))


def mcp_tool_by_name(tools: list[Any], name: str):
    for tool_item in tools:
        if tool_item.name == name:
            return tool_item
    raise KeyError(f"MCP tool을 찾을 수 없습니다: {name}")


def parse_mcp_tool_result(content: Any) -> dict[str, Any]:
    'Convert MCP content blocks or JSON strings into a dict payload.'
    if isinstance(content, dict):
        return content
    if isinstance(content, list):
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                return json.loads(item["text"])
        raise ValueError(f"MCP text content를 찾지 못했습니다: {content}")
    if isinstance(content, str):
        return json.loads(content)
    raise TypeError(f"지원하지 않는 MCP tool result 형식입니다: {type(content)}")


write_calendar_mcp_server(include_create_event=False)
calendar_mcp_tools = load_calendar_mcp_tools()
show_json([{"name": tool_item.name, "description": tool_item.description} for tool_item in calendar_mcp_tools])

check_tool = mcp_tool_by_name(calendar_mcp_tools, "calendar_check_availability")
basic_mcp_response = run_async(check_tool.ainvoke({"members": ["민수", "지아"], "date": "2026-04-24"}))
basic_mcp_payload = parse_mcp_tool_result(basic_mcp_response)
show_json(basic_mcp_payload)

mcp_agent = create_agent(
    model=make_model(600),
    tools=calendar_mcp_tools,
    system_prompt="그룹 가능 시간 질문은 실제 MCP 서버에서 로드한 calendar_check_availability 도구를 호출한 뒤 답한다.",
)

student_request = "민수와 지아가 2026-04-24에 가능한 시간 알려줘"
result = mcp_agent.invoke({"messages": [{"role": "user", "content": student_request}]})

print(final_text(result))
show_json(extract_tool_trace(result))


## 4. 카나메이트 확장 예제

같은 로컬 MCP 서버에 `calendar_create_event` tool을 추가한다. Agent는 Python 함수가 아니라 MCP 서버에서 읽어온 tool 목록으로 가능 시간 조회와 일정 생성을 모두 처리한다.


In [ ]:
write_calendar_mcp_server(include_create_event=True)
practice_mcp_tools = load_calendar_mcp_tools()
show_json([{"name": tool_item.name, "description": tool_item.description} for tool_item in practice_mcp_tools])

practice_mcp_agent = create_agent(
    model=make_model(700),
    tools=practice_mcp_tools,
    system_prompt="가능 시간 조회는 calendar_check_availability를, 일정 생성이나 확정 요청은 calendar_create_event MCP 도구를 호출한 뒤 답한다.",
)


## 5. 확장 예제 실행

확정된 그룹 일정을 만들어 달라고 요청하고, trace에 남은 실제 MCP 서버 tool result를 확인한다.


In [ ]:
practice_request = "민수와 지아의 발표 리허설을 2026-04-24 15:00 일정으로 생성해줘"
practice_result = practice_mcp_agent.invoke({"messages": [{"role": "user", "content": practice_request}]})
practice_trace = extract_tool_trace(practice_result)
create_event_payloads = [
    parse_mcp_tool_result(event["content"])
    for event in practice_trace
    if event["event"] == "tool_result" and event["tool_name"] == "calendar_create_event"
]
created_event = create_event_payloads[0]

print(final_text(practice_result))
show_json(practice_trace)
show_json(created_event)

assert any(event["event"] == "tool_call" and event["tool_name"] == "calendar_create_event" for event in practice_trace)
assert created_event["server"] == "kanamate-calendar"
assert created_event["tool"] == "calendar.create_event"
assert created_event["status"] == "created"
print("4주차 확장 예제 실행 통과")


## 6. 코드 작성형 실습(`week04.py`)

이번 실습은 실제 MCP 서버에서 tool을 로드하고 일정 생성 요청을 실행하는 helper를 같은 주차 과제 파일 `week04.py`의 `run_mcp_event_request`에서 구현하고 실행한다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "week04.py").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from week04 import extract_tool_trace, final_text, show_json, run_mcp_event_request


### 실행 예시

In [ ]:
practice_request = "민수와 지아의 발표 리허설을 2026-04-24 15:00 일정으로 생성해줘"
student_result = run_mcp_event_request(practice_request)
practice_trace = student_result["trace"]
created_event = student_result["created_event"]

print(student_result["answer"])
show_json(practice_trace)
show_json(created_event)

## 6-0. 실습 자동 점검

아래 셀은 모델 답변 문구가 아니라 trace, structured response, payload처럼 구조화된 값을 확인한다.

In [ ]:
assert created_event is not None
assert any(event["event"] == "tool_call" and event["tool_name"] == "calendar_create_event" for event in practice_trace)
assert created_event["server"] == "kanamate-calendar"
assert created_event["tool"] == "calendar.create_event"
assert {"title", "date", "start_time", "members"}.issubset(created_event["arguments"].keys())
assert created_event["event_id"]
assert created_event["status"] == "created"
print("4주차 코드 작성형 실습 통과")

## 6-1. 로컬 Gradio UI 실습

이번 주 Gradio UI도 `week04.py` 안에 들어 있다. 터미널에서 아래 명령을 실행하면 해당 주차 UI가 바로 열린다.

```bash
python week04.py
```

노트북 안에서는 다음 셀에서 `create_demo()`로 Gradio Blocks 객체를 확인할 수 있다.

In [ ]:
from week04 import create_demo

demo = create_demo()
demo

## 7. 회고

이번 주에는 MCP 스타일 JSON을 흉내내는 수준을 넘어, 로컬 MCP 서버를 직접 만들고 agent와 UI가 서버에서 로드한 tool을 호출하는 흐름을 관찰했다.
